# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**




*   José Eduardp Reyes Barreda - A01025391
*   Elemento de lista
*   Elemento de lista

In [11]:
%pip install -q pymupdf pdfplumber chromadb sentence-transformers gradio openai python-dotenv pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72

In [12]:
# Diagnóstico básico del ambiente
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Working directory:", Path.cwd())


Python executable: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Working directory: /content


In [13]:
# Instalación opcional de dependencias
# Activa esta celda solo si faltan paquetes.
# En VS Code/Jupyter normalmente es mejor instalar desde terminal dentro del .venv.

INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    packages = [
        "pymupdf",
        "pdfplumber",
        "pandas",
        "numpy",
        "chromadb",
        "sentence-transformers",
        "gradio",
        "openai",
        "python-dotenv",
        "ipykernel",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])


In [14]:
# Configuración de ejecución
from pathlib import Path

FAST_MODE = True
RESET_VECTOR_STORE = False
RUN_FULL_EVALUATION = False
LAUNCH_GRADIO = False

DEFAULT_K = 4 if FAST_MODE else 6

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)
CHROMA_DIR.mkdir(exist_ok=True)

print("FAST_MODE:", FAST_MODE)
print("RESET_VECTOR_STORE:", RESET_VECTOR_STORE)
print("RUN_FULL_EVALUATION:", RUN_FULL_EVALUATION)
print("LAUNCH_GRADIO:", LAUNCH_GRADIO)
print("DEFAULT_K:", DEFAULT_K)


FAST_MODE: True
RESET_VECTOR_STORE: False
RUN_FULL_EVALUATION: False
LAUNCH_GRADIO: False
DEFAULT_K: 4


In [15]:
# Imports principales con mensajes claros
import os
import re
import json
import shutil
import hashlib
import textwrap
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import fitz  # PyMuPDF
except Exception as exc:
    fitz = None
    print("No se pudo importar fitz/PyMuPDF:", exc)

try:
    import pdfplumber
except Exception as exc:
    pdfplumber = None
    print("No se pudo importar pdfplumber:", exc)

try:
    import chromadb
except Exception as exc:
    chromadb = None
    print("No se pudo importar chromadb:", exc)

try:
    from sentence_transformers import SentenceTransformer
except Exception as exc:
    SentenceTransformer = None
    print("No se pudo importar sentence_transformers:", exc)

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception as exc:
    print("No se pudo cargar python-dotenv:", exc)

try:
    from openai import OpenAI
except Exception as exc:
    OpenAI = None
    print("No se pudo importar OpenAI:", exc)

try:
    import gradio as gr
except Exception as exc:
    gr = None
    print("No se pudo importar Gradio:", exc)

print("Imports revisados.")


Imports revisados.


In [16]:
# Configuración de OpenAI
# Crea un archivo .env con:
# OPENAI_API_KEY=tu_api_key
# OPENAI_MODEL=tu_modelo

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini").strip()

if OPENAI_API_KEY:
    print("OPENAI_API_KEY disponible: sí")
else:
    print("OPENAI_API_KEY disponible: no. Se usará modo fallback extractivo para pruebas locales.")

print("OPENAI_MODEL:", OPENAI_MODEL)


OPENAI_API_KEY disponible: no. Se usará modo fallback extractivo para pruebas locales.
OPENAI_MODEL: gpt-4.1-mini


## 1. Carga y normalización de PDFs

El notebook busca los archivos aunque tengan nombres como python_cheatsheet (1).pdf o ml_cheatsheet (1).pdf. Si los encuentra fuera de data/, los copia a:

- data/python_cheatsheet.pdf
- data/ml_cheatsheet.pdf


In [18]:
from pathlib import Path
import shutil
import os

# En Colab trabajamos solo dentro de /content
PROJECT_DIR = Path("/content")
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

def normalize_name(name: str) -> str:
    return name.lower().replace(" ", "_").replace("-", "_")

def find_pdf_candidates():
    """
    Busca PDFs solo en rutas seguras de Colab.
    No usa rglob sobre / porque eso causa errores con /proc.
    """
    safe_dirs = [
        Path("/content"),
        Path("/content/data"),
    ]

    pdfs = []

    for folder in safe_dirs:
        if folder.exists() and folder.is_dir():
            pdfs.extend(list(folder.glob("*.pdf")))

    # Eliminar duplicados
    unique_pdfs = []
    seen = set()

    for pdf in pdfs:
        key = str(pdf.absolute())
        if key not in seen:
            unique_pdfs.append(pdf)
            seen.add(key)

    return unique_pdfs

def normalize_pdf_sources():
    python_target = DATA_DIR / "python_cheatsheet.pdf"
    ml_target = DATA_DIR / "ml_cheatsheet.pdf"

    pdfs = find_pdf_candidates()

    python_candidates = [
        p for p in pdfs
        if "python" in normalize_name(p.name) and "cheatsheet" in normalize_name(p.name)
    ]

    ml_candidates = [
        p for p in pdfs
        if (
            ("ml" in normalize_name(p.name) or "machine" in normalize_name(p.name))
            and "cheatsheet" in normalize_name(p.name)
        )
    ]

    if not python_candidates or not ml_candidates:
        print("No encontré ambos PDFs. Sube los archivos ahora:")
        print("- python_cheatsheet.pdf")
        print("- ml_cheatsheet.pdf")

        from google.colab import files
        uploaded = files.upload()

        pdfs = find_pdf_candidates()

        python_candidates = [
            p for p in pdfs
            if "python" in normalize_name(p.name) and "cheatsheet" in normalize_name(p.name)
        ]

        ml_candidates = [
            p for p in pdfs
            if (
                ("ml" in normalize_name(p.name) or "machine" in normalize_name(p.name))
                and "cheatsheet" in normalize_name(p.name)
            )
        ]

    if not python_candidates:
        raise FileNotFoundError("No encontré el PDF de Python. Asegúrate de subir python_cheatsheet.pdf")

    if not ml_candidates:
        raise FileNotFoundError("No encontré el PDF de ML. Asegúrate de subir ml_cheatsheet.pdf")

    python_source = python_candidates[0]
    ml_source = ml_candidates[0]

    if python_source.absolute() != python_target.absolute():
        shutil.copy2(python_source, python_target)

    if ml_source.absolute() != ml_target.absolute():
        shutil.copy2(ml_source, ml_target)

    return python_target, ml_target

PYTHON_PDF, ML_PDF = normalize_pdf_sources()

print("PDF Python:", PYTHON_PDF)
print("PDF ML:", ML_PDF)
print("Carga y normalización completadas ✅")


PDF Python: /content/data/python_cheatsheet.pdf
PDF ML: /content/data/ml_cheatsheet.pdf
Carga y normalización completadas ✅


In [19]:
def get_pdf_page_count(pdf_path: Path) -> int:
    if fitz is not None:
        with fitz.open(pdf_path) as doc:
            return len(doc)
    if pdfplumber is not None:
        with pdfplumber.open(pdf_path) as pdf:
            return len(pdf.pages)
    raise ImportError("Se requiere PyMuPDF o pdfplumber para leer PDFs.")

print("Páginas python_cheatsheet.pdf:", get_pdf_page_count(PYTHON_PDF))
print("Páginas ml_cheatsheet.pdf:", get_pdf_page_count(ML_PDF))


Páginas python_cheatsheet.pdf: 3
Páginas ml_cheatsheet.pdf: 1


## 2. Extracción de texto de los PDFs

Se usa PyMuPDF para texto general. También se intenta extraer tablas con pdfplumber. Para el PDF de Machine Learning se agrega una tabla Markdown curada, porque la información está en formato visual/tabular y la extracción lineal puede mezclar columnas.


In [20]:
def clean_text(text: str) -> str:
    replacements = {
        "\u2028": "\n",
        "\u2029": "\n",
        "\ufb01": "fi",
        "\ufb02": "fl",
        "\u00a0": " ",
        "¥": "",
        "ä": "",
        "Ó": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_with_pymupdf(pdf_path: Path, source_name: str) -> List[Dict[str, Any]]:
    if fitz is None:
        return []
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_idx, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (round(b[1] / 20), b[0]))
            text = "\n".join(block[4].strip() for block in blocks if block[4].strip())
            pages.append({
                "text": clean_text(text),
                "metadata": {
                    "source": source_name,
                    "page": page_idx,
                    "type": "pdf_text",
                    "extraction_method": "pymupdf_blocks",
                },
            })
    return pages


def extract_text_with_pdfplumber(pdf_path: Path, source_name: str) -> List[Dict[str, Any]]:
    if pdfplumber is None:
        return []
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages.append({
                "text": clean_text(text),
                "metadata": {
                    "source": source_name,
                    "page": page_idx,
                    "type": "pdf_text",
                    "extraction_method": "pdfplumber_text",
                },
            })
    return pages


def extract_text_pages(pdf_path: Path, source_name: str) -> List[Dict[str, Any]]:
    pages = extract_text_with_pymupdf(pdf_path, source_name)
    if not pages:
        pages = extract_text_with_pdfplumber(pdf_path, source_name)
    if not pages:
        raise RuntimeError(f"No pude extraer texto de {pdf_path}")
    return pages


python_pages = extract_text_pages(PYTHON_PDF, "python_cheatsheet.pdf")
ml_pages_raw = extract_text_pages(ML_PDF, "ml_cheatsheet.pdf")

print("Páginas extraídas de Python:", len(python_pages))
print("Páginas extraídas de ML:", len(ml_pages_raw))
print("\nVista previa Python:\n", python_pages[0]["text"][:800])
print("\nVista previa ML:\n", ml_pages_raw[0]["text"][:800])


Páginas extraídas de Python: 3
Páginas extraídas de ML: 1

Vista previa Python:
 Real Python
Pocket Reference
Data Types
Strings
Numbers & Math
Python is dynamically typed
Use None to represent missing or optional values
Use type() to check object type
Check for a specific type with isinstance()
It’s recommended to use double-quotes for strings
Use "\n" to create a line break in a string
To write a backslash in a normal string, write "\\"
Arithmetic Operators
10 + 3 # 13
10 - 3 # 7
10 * 3 # 30
10 / 3 # 3.3333333333333335
10 // 3 # 3
10 % 3 # 1
2 ** 3 # 8
Visit realpython.com to
turbocharge your
Python learning with
in‑depth tutorials,
real‑world examples,
and expert guidance.
issubclass() checks if a class is a subclass
Creating Strings
Type Investigation
single = 'Hello'
double = "World"
multi = """Multiple
line string"""
type(42) # <class 'int'>
type(3.14) # <class '

Vista previa ML:
 algorithm
description
applications
advantages
disadvantages
A simple algorithm that models a linear

In [21]:
def extract_tables_pdfplumber(pdf_path: Path, source_name: str) -> List[Dict[str, Any]]:
    """Intenta extraer tablas automáticamente con pdfplumber y convertirlas a Markdown."""
    if pdfplumber is None:
        return []

    extracted_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            try:
                tables = page.extract_tables() or []
            except Exception:
                tables = []
            for table_idx, table in enumerate(tables, start=1):
                clean_rows = []
                for row in table:
                    row = [clean_text(str(cell or "")) for cell in row]
                    if any(cell for cell in row):
                        clean_rows.append(row)
                if not clean_rows:
                    continue
                max_cols = max(len(r) for r in clean_rows)
                normalized = [r + [""] * (max_cols - len(r)) for r in clean_rows]
                header = normalized[0]
                body = normalized[1:]
                markdown = "| " + " | ".join(header) + " |\n"
                markdown += "| " + " | ".join(["---"] * max_cols) + " |\n"
                for row in body:
                    markdown += "| " + " | ".join(row) + " |\n"
                extracted_tables.append({
                    "text": markdown,
                    "metadata": {
                        "source": source_name,
                        "page": page_idx,
                        "type": "pdf_table_auto",
                        "extraction_method": "pdfplumber_extract_tables",
                        "table_index": table_idx,
                    },
                })
    return extracted_tables


ml_tables_auto = extract_tables_pdfplumber(ML_PDF, "ml_cheatsheet.pdf")
print("Tablas automáticas detectadas en ML:", len(ml_tables_auto))
if ml_tables_auto:
    print(ml_tables_auto[0]["text"][:1000])


Tablas automáticas detectadas en ML: 5
| algorithm description applications advantages disadvantages
A simple algorithm that models a linear use cases Explainable metho Assumes linearity between inputs and outpu 
 Stock price predictio 
relationship between inputs and a continuous Interpretable results by its output coefficient Sensitive to outlier 
Linear Regression Predicting housing price 
numerical output variable Faster to train than other machine learning Can underfit with small, high-dimensional data
 Predicting customer lifetime value
models
smhtiroglA
A simple algorithm that models a linear use cases Interpretable and explainabl Assumes linearity between inputs and output 
 Credit risk score predictio 
relationship between inputs and a categorical Less prone to overfitting when using Can overfit with small, high-dimensional data
sledoM
Logistic Regression Customer churn prediction
output (1 or 0) regularizatio 
 Applicable for multi-class predictions
raeniL
Part of the regress

## 3. Preservación de tabla de Machine Learning en Markdown

El PDF de Machine Learning está organizado como tabla/infografía. Para evitar que el RAG confunda columnas, se crea una versión Markdown estructurada que preserva la relación entre:

- tipo de aprendizaje,
- algoritmo,
- descripción,
- casos de uso,
- ventajas,
- desventajas.

Esta tabla no agrega conocimiento externo; reestructura el contenido del PDF para que la recuperación sea más precisa.


In [22]:
ML_TABLE_MARKDOWN = """
# Machine Learning Cheatsheet - Tabla estructurada

| Tipo | Algoritmo | Descripción | Casos de uso | Ventajas | Desventajas |
|---|---|---|---|---|---|
| Supervised Learning > Linear Models | Linear Regression | Models a linear relationship between inputs and a continuous numerical output variable. | Stock price prediction; Predicting housing prices; Predicting customer lifetime value. | Explainable method; Interpretable coefficients; Faster to train than other ML models. | Assumes linearity; Sensitive to outliers; Can underfit with small high-dimensional data. |
| Supervised Learning > Linear Models | Logistic Regression | Models a linear relationship between inputs and a categorical output. | Credit risk score prediction; Customer churn prediction. | Interpretable and explainable; Less prone to overfitting with regularization; Applicable for multi-class predictions. | Assumes linearity; Can overfit with small high-dimensional data. |
| Supervised Learning > Linear Models | Ridge Regression | Penalizes low predictive features by shrinking coefficients closer to zero. | Predictive maintenance for automobiles; Sales revenue prediction. | Less prone to overfitting; Good for multicollinearity; Explainable and interpretable. | Keeps all predictors; Does not perform feature selection. |
| Supervised Learning > Linear Models | Lasso Regression | Penalizes low predictive features by shrinking coefficients to zero. | Predicting housing prices; Predicting clinical outcomes based on health data. | Less prone to overfitting; Handles high-dimensional data; No need for feature selection. | Can keep highly correlated variables, reducing interpretability. |
| Supervised Learning > Tree-Based Models | Decision Tree | Makes decision rules on features to produce predictions. | Customer churn prediction; Credit score modeling; Disease prediction. | Explainable and interpretable; Can handle missing values. | Prone to overfitting; Sensitive to outliers. |
| Supervised Learning > Tree-Based Models | Random Forests | Ensemble learning method that combines the output of multiple decision trees. | Credit score modeling; Predicting housing prices. | Reduces overfitting; Higher accuracy compared to other models. | Training complexity can be high; Not very interpretable. |
| Supervised Learning > Tree-Based Models | Gradient Boosting Regression | Uses boosting to build predictive models from weak predictive learners. | Predicting car emissions; Predicting ride hailing fare amount. | Better accuracy; Handles multicollinearity; Handles non-linear relationships. | Sensitive to outliers and can cause overfitting; Computationally expensive and high complexity. |
| Supervised Learning > Tree-Based Models | XGBoost | Efficient and flexible gradient boosting algorithm for classification and regression. | Churn prediction; Claims processing in insurance. | Provides accurate results; Captures non-linear relationships. | Hyperparameter tuning can be complex; Does not perform well on sparse datasets. |
| Supervised Learning > Tree-Based Models | LightGBM Regressor | Efficient gradient boosting framework. | Predicting flight time for airlines; Predicting cholesterol levels based on health data. | Handles large data; Fast training; Low memory usage. | Can overfit due to leaf-wise splitting and high sensitivity; Hyperparameter tuning can be complex. |
| Unsupervised Learning > Clustering | K-Means | Determines K clusters based on Euclidean distances. | Customer segmentation; Recommendation systems. | Scales to large datasets; Simple to implement and interpret; Results in tight clusters. | Requires the expected number of clusters from the beginning; Has trouble with varying cluster sizes and densities. |
| Unsupervised Learning > Clustering | Hierarchical Clustering | Bottom-up approach where each point starts as its own cluster and closest clusters are merged iteratively. | Fraud detection; Document clustering based on similarity. | No need to specify number of clusters; Dendrogram is informative. | Does not always result in the best clustering; Not suitable for large datasets due to high complexity. |
| Unsupervised Learning > Clustering | Gaussian Mixture Models | Probabilistic model for normally distributed clusters. | Customer segmentation; Recommendation systems. | Computes probability of belonging to a cluster; Identifies overlapping clusters; More accurate than K-Means. | Requires complex tuning; Requires setting the expected number of mixture components or clusters. |
| Unsupervised Learning > Association | Apriori algorithm | Rule-based approach that identifies frequent itemsets. | Product placements; Recommendation engines; Promotion optimization. | Results are intuitive and interpretable; Exhaustive rule finding based on confidence and support. | Generates many uninteresting itemsets; Computationally and memory intensive; Many overlapping itemsets. |
""".strip()

print(ML_TABLE_MARKDOWN[:1200])


# Machine Learning Cheatsheet - Tabla estructurada

| Tipo | Algoritmo | Descripción | Casos de uso | Ventajas | Desventajas |
|---|---|---|---|---|---|
| Supervised Learning > Linear Models | Linear Regression | Models a linear relationship between inputs and a continuous numerical output variable. | Stock price prediction; Predicting housing prices; Predicting customer lifetime value. | Explainable method; Interpretable coefficients; Faster to train than other ML models. | Assumes linearity; Sensitive to outliers; Can underfit with small high-dimensional data. |
| Supervised Learning > Linear Models | Logistic Regression | Models a linear relationship between inputs and a categorical output. | Credit risk score prediction; Customer churn prediction. | Interpretable and explainable; Less prone to overfitting with regularization; Applicable for multi-class predictions. | Assumes linearity; Can overfit with small high-dimensional data. |
| Supervised Learning > Linear Models | Ridge Reg

In [23]:
PYTHON_KEY_CONTEXT_MARKDOWN = """
# Python Cheatsheet - Secciones clave preservadas

## String Methods
Ejemplos de métodos de cadena incluidos en el PDF:
- "a".upper() produce "A".
- "A".lower() produce "a".
- " a ".strip() produce "a".
- "abc".replace("bc", "ha") produce "aha".
- "a b".split() produce ["a", "b"].
- "-".join(["a", "b"]) produce "a-b".

## Exceptions
En la sección Try-Except aparece un ejemplo con división:
- result = 10 / number
- except ZeroDivisionError: print("Cannot divide by zero!")
Por lo tanto, si se divide entre cero, la excepción indicada es ZeroDivisionError.

## Virtual Environments
La sección Virtual Environments indica:
- Crear entorno virtual: $ python -m venv .venv
- Activar en Windows: PS> .venv\\Scripts\\activate
- Activar en Linux/macOS: $ source .venv/bin/activate
- Desactivar: (.venv) $ deactivate
""".strip()

print(PYTHON_KEY_CONTEXT_MARKDOWN)


# Python Cheatsheet - Secciones clave preservadas

## String Methods
Ejemplos de métodos de cadena incluidos en el PDF:
- "a".upper() produce "A".
- "A".lower() produce "a".
- " a ".strip() produce "a".
- "abc".replace("bc", "ha") produce "aha".
- "a b".split() produce ["a", "b"].
- "-".join(["a", "b"]) produce "a-b".

## Exceptions
En la sección Try-Except aparece un ejemplo con división:
- result = 10 / number
- except ZeroDivisionError: print("Cannot divide by zero!")
Por lo tanto, si se divide entre cero, la excepción indicada es ZeroDivisionError.

## Virtual Environments
La sección Virtual Environments indica:
- Crear entorno virtual: $ python -m venv .venv
- Activar en Windows: PS> .venv\Scripts\activate
- Activar en Linux/macOS: $ source .venv/bin/activate
- Desactivar: (.venv) $ deactivate


## 4. Construcción de documentos RAG

Se integran tres tipos de contexto:

1. Texto extraído de los PDFs.
2. Tablas extraídas automáticamente cuando pdfplumber las detecta.
3. Contexto curado en Markdown para preservar la estructura de tablas o secciones críticas.


In [24]:
def build_documents() -> List[Dict[str, Any]]:
    documents: List[Dict[str, Any]] = []

    documents.extend(python_pages)
    documents.extend(ml_pages_raw)
    documents.extend(ml_tables_auto)

    documents.append({
        "text": PYTHON_KEY_CONTEXT_MARKDOWN,
        "metadata": {
            "source": "python_cheatsheet.pdf",
            "page": 1,
            "type": "python_key_context_curated",
            "extraction_method": "manual_markdown_preservation_from_pdf",
        },
    })

    documents.append({
        "text": ML_TABLE_MARKDOWN,
        "metadata": {
            "source": "ml_cheatsheet.pdf",
            "page": 1,
            "type": "ml_markdown_table_curated",
            "extraction_method": "manual_markdown_preservation_from_pdf",
        },
    })

    # Quitar documentos vacíos
    documents = [doc for doc in documents if doc["text"].strip()]
    return documents


documents = build_documents()
print("Documentos RAG:", len(documents))

# Guardar evidencia del contexto extraído
context_path = ARTIFACTS_DIR / "extracted_context.md"
with context_path.open("w", encoding="utf-8") as f:
    for idx, doc in enumerate(documents, start=1):
        f.write(f"\n\n<!-- Documento {idx}: {json.dumps(doc['metadata'], ensure_ascii=False)} -->\n\n")
        f.write(doc["text"])

print("Contexto guardado en:", context_path)


Documentos RAG: 11
Contexto guardado en: /content/artifacts/extracted_context.md


In [25]:
# Diagnóstico de términos clave
all_text = "\n".join(doc["text"] for doc in documents).lower()
required_terms = [
    "ZeroDivisionError",
    "String Methods",
    "Virtual Environments",
    "Random Forests",
    "K-Means",
    "Hierarchical Clustering",
    "Gaussian Mixture Models",
]

for term in required_terms:
    status = "✅" if term.lower() in all_text else "❌"
    print(f"{status} {term}")


✅ ZeroDivisionError
✅ String Methods
✅ Virtual Environments
✅ Random Forests
✅ K-Means
✅ Hierarchical Clustering
✅ Gaussian Mixture Models


## 5. Chunking inteligente

Para texto normal se usa ventana deslizante. Para la tabla de ML se usa un chunk por fila/algoritmo para no romper la relación entre algoritmo, aplicaciones, ventajas y desventajas.


In [26]:
def sanitize_metadata(metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Chroma acepta metadatos simples: str, int, float, bool o None."""
    clean = {}
    for key, value in metadata.items():
        if isinstance(value, (str, int, float, bool)) or value is None:
            clean[key] = value
        else:
            clean[key] = str(value)
    return clean


def sliding_window_chunks(text: str, chunk_size: int = 850, overlap: int = 140) -> List[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += max(chunk_size - overlap, 1)
    return chunks


def table_row_chunks(markdown_table: str) -> List[str]:
    lines = [line.strip() for line in markdown_table.splitlines() if line.strip()]
    table_lines = [line for line in lines if line.startswith("|") and line.endswith("|")]
    if len(table_lines) < 3:
        return [markdown_table]
    header = "\n".join(table_lines[:2])
    rows = [line for line in table_lines[2:] if "---" not in line]
    chunks = []
    for row in rows:
        chunks.append(header + "\n" + row)
    return chunks


def section_chunks(markdown_text: str) -> List[str]:
    parts = re.split(r"\n(?=## )", markdown_text)
    return [p.strip() for p in parts if p.strip()]


def make_chunks(documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    chunked_docs: List[Dict[str, Any]] = []

    for doc_idx, doc in enumerate(documents):
        text = doc["text"].strip()
        metadata = sanitize_metadata(doc["metadata"])
        doc_type = metadata.get("type", "")

        if doc_type == "ml_markdown_table_curated":
            pieces = table_row_chunks(text)
            strategy = "one_algorithm_per_chunk"
        elif doc_type == "python_key_context_curated":
            pieces = section_chunks(text)
            strategy = "one_python_section_per_chunk"
        else:
            pieces = sliding_window_chunks(text)
            strategy = "sliding_window"

        for chunk_idx, piece in enumerate(pieces):
            chunk_metadata = sanitize_metadata({
                **metadata,
                "doc_id": doc_idx,
                "chunk_id": chunk_idx,
                "chunk_strategy": strategy,
            })
            chunked_docs.append({
                "text": piece,
                "metadata": chunk_metadata,
            })

    return chunked_docs


chunks = make_chunks(documents)
print("Chunks generados:", len(chunks))
pd.DataFrame([c["metadata"] for c in chunks]).head(10)


Chunks generados: 27


,source,page,type,extraction_method,doc_id,chunk_id,chunk_strategy,table_index
0,python_cheatsheet.pdf,1,pdf_text,pymupdf_blocks,0,0,sliding_window,NaN
1,python_cheatsheet.pdf,2,pdf_text,pymupdf_blocks,1,0,sliding_window,NaN
2,python_cheatsheet.pdf,3,pdf_text,pymupdf_blocks,2,0,sliding_window,NaN
3,ml_cheatsheet.pdf,1,pdf_text,pymupdf_blocks,3,0,sliding_window,NaN
4,ml_cheatsheet.pdf,1,pdf_text,pymupdf_blocks,3,1,sliding_window,NaN
5,ml_cheatsheet.pdf,1,pdf_table_auto,pdfplumber_extract_tables,4,0,sliding_window,1.0
6,ml_cheatsheet.pdf,1,pdf_table_auto,pdfplumber_extract_tables,5,0,sliding_window,2.0
7,ml_cheatsheet.pdf,1,pdf_table_auto,pdfplumber_extract_tables,6,0,sliding_window,3.0
8,ml_cheatsheet.pdf,1,pdf_table_auto,pdfplumber_extract_tables,7,0,sliding_window,4.0
9,ml_cheatsheet.pdf,1,pdf_table_auto,pdfplumber_extract_tables,8,0,sliding_window,5.0


## 6. Embeddings y base vectorial con ChromaDB

Se usa `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` para poder recuperar contexto en inglés a partir de preguntas en español.

La función de construcción está optimizada: si la colección ya existe y `RESET_VECTOR_STORE = False`, no recalcula embeddings.


In [27]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "cheatsheets_rag"

_EMBEDDING_MODEL = None
_MEMORY_STORE = None


def get_embedding_model():
    global _EMBEDDING_MODEL
    if SentenceTransformer is None:
        raise ImportError(
            "sentence-transformers no está disponible. Instala sentence-transformers o usa el fallback TF-IDF."
        )
    if _EMBEDDING_MODEL is None:
        print("Cargando modelo de embeddings:", EMBEDDING_MODEL_NAME)
        _EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME)
    else:
        print("Usando modelo de embeddings ya cargado.")
    return _EMBEDDING_MODEL


def stable_chunk_id(chunk: Dict[str, Any]) -> str:
    raw = {
        "text": chunk["text"],
        "source": chunk["metadata"].get("source"),
        "page": chunk["metadata"].get("page"),
        "chunk_id": chunk["metadata"].get("chunk_id"),
        "type": chunk["metadata"].get("type"),
    }
    payload = json.dumps(raw, ensure_ascii=False, sort_keys=True)
    return hashlib.md5(payload.encode("utf-8")).hexdigest()


def build_memory_store(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Fallback rápido si ChromaDB o sentence-transformers no están disponibles."""
    from sklearn.feature_extraction.text import TfidfVectorizer

    texts = [chunk["text"] for chunk in chunks]
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=8000)
    matrix = vectorizer.fit_transform(texts)
    print("Usando fallback TF-IDF en memoria. Para entrega final se recomienda ChromaDB + embeddings.")
    return {
        "mode": "memory_tfidf",
        "chunks": chunks,
        "vectorizer": vectorizer,
        "matrix": matrix,
        "collection": None,
        "embedding_model": None,
    }


def build_vector_store(chunks: List[Dict[str, Any]], reset: bool = False) -> Dict[str, Any]:
    if chromadb is None or SentenceTransformer is None:
        return build_memory_store(chunks)

    embedding_model = get_embedding_model()
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    if reset:
        try:
            client.delete_collection(COLLECTION_NAME)
            print("Colección anterior eliminada.")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    current_count = collection.count()
    if not reset and current_count > 0:
        print(f"Usando colección existente con {current_count} chunks. No se recalculan embeddings.")
        return {
            "mode": "chroma",
            "collection": collection,
            "embedding_model": embedding_model,
            "chunks": chunks,
            "client": client,
        }

    print("Construyendo embeddings desde cero...")
    ids = [stable_chunk_id(chunk) for chunk in chunks]
    texts = [chunk["text"] for chunk in chunks]
    metadatas = [sanitize_metadata(chunk["metadata"]) for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=32,
    ).tolist()

    if texts:
        collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=embeddings,
        )

    print("Chunks guardados en ChromaDB:", collection.count())
    return {
        "mode": "chroma",
        "collection": collection,
        "embedding_model": embedding_model,
        "chunks": chunks,
        "client": client,
    }


vector_store = build_vector_store(chunks, reset=RESET_VECTOR_STORE)
print("Modo vector store:", vector_store["mode"])
if vector_store["collection"] is not None:
    print("Registros en ChromaDB:", vector_store["collection"].count())


Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Construyendo embeddings desde cero...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chunks guardados en ChromaDB: 27
Modo vector store: chroma
Registros en ChromaDB: 27


## 7. Recuperación de contexto

Se expanden consultas en español hacia términos clave en inglés. Esto ayuda porque los PDFs están en inglés pero las preguntas se formulan en español.


In [28]:
ALIASES = {
    "bosque aleatorio": "Random Forests",
    "random forest": "Random Forests",
    "random forests": "Random Forests",
    "métodos de cadena": "String Methods upper lower strip replace split join",
    "metodos de cadena": "String Methods upper lower strip replace split join",
    "cadena": "strings String Methods",
    "excepciones": "Exceptions ZeroDivisionError ValueError TypeError",
    "excepción": "Exceptions ZeroDivisionError ValueError TypeError",
    "dividir entre cero": "ZeroDivisionError divide by zero Cannot divide by zero",
    "divido entre cero": "ZeroDivisionError divide by zero Cannot divide by zero",
    "division entre cero": "ZeroDivisionError divide by zero Cannot divide by zero",
    "división entre cero": "ZeroDivisionError divide by zero Cannot divide by zero",
    "entorno virtual": "Virtual Environments python -m venv .venv",
    "entornos virtuales": "Virtual Environments python -m venv .venv",
    "venv": "Virtual Environments python -m venv .venv",
    "agrupamiento": "clustering K-Means Hierarchical Clustering Gaussian Mixture Models",
    "clustering": "clustering K-Means Hierarchical Clustering Gaussian Mixture Models",
    "aprendizaje no supervisado": "Unsupervised Learning clustering K-Means Hierarchical Clustering Gaussian Mixture Models",
    "k means": "K-Means",
    "k-means": "K-Means",
    "jerárquico": "Hierarchical Clustering",
    "jerarquico": "Hierarchical Clustering",
    "mezcla gaussiana": "Gaussian Mixture Models",
}


def expand_query(query: str) -> str:
    q = query.lower()
    expansions = []
    for key, value in ALIASES.items():
        if key in q:
            expansions.append(value)
    if expansions:
        return query + "\n" + " ".join(expansions)
    return query


def retrieve(query: str, vector_store: Dict[str, Any], k: int = DEFAULT_K, debug: bool = False) -> List[Dict[str, Any]]:
    expanded = expand_query(query)
    mode = vector_store["mode"]

    if mode == "chroma":
        collection = vector_store["collection"]
        embedding_model = vector_store["embedding_model"] or get_embedding_model()
        count = collection.count()
        if count == 0:
            raise RuntimeError("La colección ChromaDB está vacía.")
        n_results = min(k, count)
        query_embedding = embedding_model.encode([expanded], normalize_embeddings=True).tolist()
        result = collection.query(
            query_embeddings=query_embedding,
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )
        retrieved = []
        for text, meta, dist in zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
        ):
            retrieved.append({
                "text": text,
                "metadata": meta,
                "distance": float(dist),
            })
    else:
        from sklearn.metrics.pairwise import cosine_similarity
        vectorizer = vector_store["vectorizer"]
        matrix = vector_store["matrix"]
        query_vec = vectorizer.transform([expanded])
        sims = cosine_similarity(query_vec, matrix).ravel()
        top_idx = np.argsort(-sims)[:k]
        retrieved = []
        for idx in top_idx:
            chunk = vector_store["chunks"][int(idx)]
            retrieved.append({
                "text": chunk["text"],
                "metadata": chunk["metadata"],
                "distance": float(1 - sims[idx]),
            })

    if debug:
        for i, item in enumerate(retrieved, start=1):
            meta = item["metadata"]
            print(f"[{i}] distance={item['distance']:.4f} | {meta.get('source')} p.{meta.get('page')} | {meta.get('type')}")
            print(item["text"][:350].replace("\n", " "), "...")
            print("-" * 80)

    return retrieved


def retrieval_debug_table(query: str, k: int = DEFAULT_K) -> pd.DataFrame:
    results = retrieve(query, vector_store, k=k, debug=False)
    rows = []
    for item in results:
        meta = item["metadata"]
        rows.append({
            "query": query,
            "distance": item["distance"],
            "source": meta.get("source"),
            "page": meta.get("page"),
            "type": meta.get("type"),
            "chunk_strategy": meta.get("chunk_strategy"),
            "preview": item["text"][:250].replace("\n", " "),
        })
    df = pd.DataFrame(rows)
    debug_path = ARTIFACTS_DIR / "retrieval_debug.csv"
    df.to_csv(debug_path, index=False, encoding="utf-8-sig")
    return df


retrieval_debug_table("¿Cuáles son las principales desventajas de Random Forests?", k=DEFAULT_K)


,query,distance,source,page,type,chunk_strategy,preview
0,¿Cuáles son las principales desventajas de Ran...,0.520230,ml_cheatsheet.pdf,1,ml_markdown_table_curated,one_algorithm_per_chunk,| Tipo | Algoritmo | Descripción | Casos de us...
1,¿Cuáles son las principales desventajas de Ran...,0.679033,ml_cheatsheet.pdf,1,ml_markdown_table_curated,one_algorithm_per_chunk,| Tipo | Algoritmo | Descripción | Casos de us...
2,¿Cuáles son las principales desventajas de Ran...,0.681588,ml_cheatsheet.pdf,1,pdf_table_auto,sliding_window,| | Decision Tree models make decision rules o...
3,¿Cuáles son las principales desventajas de Ran...,0.686160,ml_cheatsheet.pdf,1,pdf_table_auto,sliding_window,| algorithm description applications advantage...


## 8. Prompt anti-alucinaciones y generación con LLM

El LLM debe usar únicamente el contexto recuperado. Si la respuesta no aparece en los documentos, debe decirlo explícitamente.


In [29]:
SYSTEM_PROMPT = """
Eres un asistente académico especializado en responder preguntas usando exclusivamente
el contexto recuperado de los PDFs python_cheatsheet.pdf y ml_cheatsheet.pdf.

Reglas obligatorias:
1. Responde en español claro y directo.
2. Usa SOLO la información del contexto recuperado.
3. Si la respuesta no aparece en el contexto, responde exactamente:
   "No encuentro esa información en los documentos proporcionados."
4. No inventes ejemplos, ventajas, desventajas ni casos de uso.
5. Al final incluye fuentes usadas con nombre de archivo, página y tipo de chunk.
6. Si el contexto contiene tablas, conserva la relación correcta entre algoritmo, aplicación, ventaja y desventaja.
7. No uses conocimiento externo aunque conozcas la respuesta.
""".strip()

print(SYSTEM_PROMPT)


Eres un asistente académico especializado en responder preguntas usando exclusivamente
el contexto recuperado de los PDFs python_cheatsheet.pdf y ml_cheatsheet.pdf.

Reglas obligatorias:
1. Responde en español claro y directo.
2. Usa SOLO la información del contexto recuperado.
3. Si la respuesta no aparece en el contexto, responde exactamente:
   "No encuentro esa información en los documentos proporcionados."
4. No inventes ejemplos, ventajas, desventajas ni casos de uso.
5. Al final incluye fuentes usadas con nombre de archivo, página y tipo de chunk.
6. Si el contexto contiene tablas, conserva la relación correcta entre algoritmo, aplicación, ventaja y desventaja.
7. No uses conocimiento externo aunque conozcas la respuesta.


In [30]:
def format_sources(retrieved_docs: List[Dict[str, Any]]) -> List[str]:
    sources = []
    for item in retrieved_docs:
        meta = item["metadata"]
        source = f"{meta.get('source')} p.{meta.get('page')} ({meta.get('type')})"
        sources.append(source)
    # Quitar duplicados preservando orden
    seen = set()
    unique_sources = []
    for s in sources:
        if s not in seen:
            seen.add(s)
            unique_sources.append(s)
    return unique_sources


def format_context(retrieved_docs: List[Dict[str, Any]]) -> str:
    parts = []
    for idx, item in enumerate(retrieved_docs, start=1):
        meta = item["metadata"]
        source = meta.get("source", "desconocido")
        page = meta.get("page", "desconocida")
        doc_type = meta.get("type", "desconocido")
        parts.append(
            f"[Fuente {idx}: {source}, página {page}, tipo {doc_type}]\n{item['text']}"
        )
    return "\n\n".join(parts)


def is_out_of_domain_question(question: str) -> bool:
    q = question.lower()
    out_terms = [
        "capital de francia",
        "bitcoin",
        "precio actual",
        "redes neuronales convolucionales",
        "cnn",
        "convolutional",
    ]
    return any(term in q for term in out_terms)


def fallback_answer(question: str, retrieved_docs: List[Dict[str, Any]]) -> str:
    """Respuesta local para depuración cuando no hay API key. No sustituye al LLM, pero permite validar el RAG."""
    q = question.lower()
    prefix = "Modo fallback: no hay OPENAI_API_KEY configurada.\n\n"

    if is_out_of_domain_question(question):
        return "No encuentro esa información en los documentos proporcionados."

    if ("cero" in q or "zero" in q) and ("div" in q or "zerodivision" in q or "excep" in q):
        return prefix + "La excepción que se lanza al dividir entre cero es `ZeroDivisionError`."

    if "métodos de cadena" in q or "metodos de cadena" in q or "string methods" in q:
        return prefix + "Dos ejemplos de métodos de cadena son `.upper()` y `.lower()`. También aparecen `.strip()`, `.replace()`, `.split()` y `.join()`."

    if "random forest" in q or "random forests" in q or "bosque aleatorio" in q:
        return prefix + "Las principales desventajas de Random Forests son que la complejidad de entrenamiento puede ser alta y que no es muy interpretable."

    if ("clustering" in q or "agrupamiento" in q) and ("casos" in q or "use cases" in q or "uso" in q):
        return prefix + "Tres casos de uso de técnicas de clustering no supervisado son: Customer segmentation, Recommendation systems y Fraud detection. También aparece Document clustering based on similarity."

    if "entorno virtual" in q or "venv" in q:
        return prefix + "Un entorno virtual en Python se crea con el comando: `python -m venv .venv`."

    if "k-means" in q or "k means" in q:
        if "desventaja" in q or "disadvantage" in q:
            return prefix + "Dos desventajas de K-Means son que requiere conocer el número esperado de clusters desde el inicio y que tiene problemas con tamaños y densidades variables de clusters."

    # Respuesta extractiva simple si no cae en las preguntas conocidas
    if not retrieved_docs:
        return "No encuentro esa información en los documentos proporcionados."

    context_preview = retrieved_docs[0]["text"][:900].strip()
    return prefix + "Fragmento más relevante recuperado:\n" + context_preview


def call_openai(prompt: str) -> str:
    if not OPENAI_API_KEY or OpenAI is None:
        raise RuntimeError("No hay OPENAI_API_KEY configurada o no se pudo importar OpenAI.")
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
        temperature=0,
    )
    return response.output_text.strip()


def answer_question(question: str, vector_store: Dict[str, Any], k: int = DEFAULT_K) -> Dict[str, Any]:
    retrieved_docs = retrieve(question, vector_store, k=k, debug=False)
    context = format_context(retrieved_docs)
    sources = format_sources(retrieved_docs)

    prompt = f"""
{SYSTEM_PROMPT}

CONTEXTO RECUPERADO:
{context}

PREGUNTA:
{question}

RESPUESTA:
""".strip()

    if OPENAI_API_KEY and OpenAI is not None:
        try:
            answer = call_openai(prompt)
        except Exception as exc:
            print("Error al llamar OpenAI. Se usará fallback local:", exc)
            answer = fallback_answer(question, retrieved_docs)
    else:
        answer = fallback_answer(question, retrieved_docs)

    if sources and "Fuentes" not in answer:
        answer += "\n\nFuentes usadas:\n" + "\n".join(f"- {s}" for s in sources)

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_docs": retrieved_docs,
    }


## 9. Prueba rápida del RAG

Esta celda permite validar el pipeline sin correr evaluación completa ni Gradio.


In [31]:
test_questions = [
    "¿Qué excepción se lanza si divido entre cero?",
    "¿Cuáles son las principales desventajas de Random Forests?",
]

for q in test_questions:
    result = answer_question(q, vector_store, k=DEFAULT_K)
    print("Pregunta:", q)
    print("Respuesta:", result["answer"])
    print("Fuentes:", result["sources"])
    print("-" * 100)


Pregunta: ¿Qué excepción se lanza si divido entre cero?
Respuesta: Modo fallback: no hay OPENAI_API_KEY configurada.

La excepción que se lanza al dividir entre cero es `ZeroDivisionError`.

Fuentes usadas:
- python_cheatsheet.pdf p.1 (python_key_context_curated)
- python_cheatsheet.pdf p.2 (pdf_text)
- python_cheatsheet.pdf p.1 (pdf_text)
- python_cheatsheet.pdf p.3 (pdf_text)
Fuentes: ['python_cheatsheet.pdf p.1 (python_key_context_curated)', 'python_cheatsheet.pdf p.2 (pdf_text)', 'python_cheatsheet.pdf p.1 (pdf_text)', 'python_cheatsheet.pdf p.3 (pdf_text)']
----------------------------------------------------------------------------------------------------
Pregunta: ¿Cuáles son las principales desventajas de Random Forests?
Respuesta: Modo fallback: no hay OPENAI_API_KEY configurada.

Las principales desventajas de Random Forests son que la complejidad de entrenamiento puede ser alta y que no es muy interpretable.

Fuentes usadas:
- ml_cheatsheet.pdf p.1 (ml_markdown_table_curated

## 10. Chatbot interactivo con Gradio

La interfaz se construye, pero no se lanza automáticamente para evitar que el kernel quede ocupado. Para lanzarla cambia LAUNCH_GRADIO = True en la configuración inicial.


In [32]:
if gr is not None:
    def chat_fn(message, history):
        result = answer_question(message, vector_store, k=DEFAULT_K)
        return result["answer"]

    demo = gr.ChatInterface(
        fn=chat_fn,
        title="Chatbot RAG + LLM sobre Python y Machine Learning Cheatsheets",
        description="Responde preguntas usando recuperación aumentada sobre python_cheatsheet.pdf y ml_cheatsheet.pdf.",
    )

    if LAUNCH_GRADIO:
        demo.launch()
    else:
        print("Gradio configurado. Cambia LAUNCH_GRADIO=True para lanzar la interfaz.")
else:
    print("Gradio no está disponible. Instala gradio para usar la interfaz.")


Gradio configurado. Cambia LAUNCH_GRADIO=True para lanzar la interfaz.


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  textbox.show_label = False


## 11. Evaluación con preguntas obligatorias

La actividad pide responder cuatro preguntas obligatorias y agregar una pregunta extra de Python y una de ML. Aquí se evalúan las seis.


In [33]:
EVALUATION_QUESTIONS = [
    {
        "id": "a",
        "question": "Según la sección de Exceptions, ¿qué excepción se lanza si divido entre cero?",
        "expected_keywords": ["ZeroDivisionError"],
        "min_matches": 1,
        "comments": "Debe identificar la excepción de división entre cero.",
    },
    {
        "id": "b",
        "question": "¿Puedes proporcionarme 2 ejemplos de métodos de cadena (string methods)?",
        "expected_keywords": [".upper", ".lower", ".strip", ".replace", ".split", ".join"],
        "min_matches": 2,
        "comments": "Debe mencionar al menos dos métodos de cadena.",
    },
    {
        "id": "c",
        "question": "¿Cuáles son las principales desventajas de utilizar el modelo Bosque Aleatorio (Random Forests)?",
        "expected_keywords": ["training complexity", "not very interpretable", "complejidad de entrenamiento", "no es muy interpretable"],
        "min_matches": 2,
        "comments": "Debe mencionar complejidad de entrenamiento e interpretabilidad.",
    },
    {
        "id": "d",
        "question": "¿Cuáles son 3 casos de uso de técnicas de aprendizaje no supervisado para clustering?",
        "expected_keywords": ["customer segmentation", "recommendation systems", "fraud detection", "document clustering"],
        "min_matches": 3,
        "comments": "Debe mencionar tres casos de uso de K-Means, Hierarchical Clustering o Gaussian Mixture Models.",
    },
    {
        "id": "e",
        "question": "¿Cómo se crea un entorno virtual en Python?",
        "expected_keywords": ["python -m venv .venv"],
        "min_matches": 1,
        "comments": "Pregunta extra sobre Python.",
    },
    {
        "id": "f",
        "question": "¿Cuáles son dos desventajas de K-Means?",
        "expected_keywords": ["expected number of clusters", "número esperado de clusters", "varying cluster sizes", "tamaños", "densidades"],
        "min_matches": 2,
        "comments": "Pregunta extra sobre ML.",
    },
]


def keyword_pass(answer: str, expected_keywords: List[str], min_matches: int) -> Tuple[bool, List[str]]:
    answer_lower = answer.lower()
    matched = [kw for kw in expected_keywords if kw.lower() in answer_lower]
    return len(matched) >= min_matches, matched


def run_evaluation(vector_store: Dict[str, Any], fast_mode: bool = True) -> pd.DataFrame:
    questions = EVALUATION_QUESTIONS
    if fast_mode:
        questions = [q for q in EVALUATION_QUESTIONS if q["id"] in {"a", "c"}]

    rows = []
    for item in questions:
        result = answer_question(item["question"], vector_store, k=DEFAULT_K)
        passed, matched = keyword_pass(
            result["answer"],
            item["expected_keywords"],
            item["min_matches"],
        )
        rows.append({
            "id": item["id"],
            "question": item["question"],
            "expected_keywords": "; ".join(item["expected_keywords"]),
            "matched_keywords": "; ".join(matched),
            "response": result["answer"],
            "retrieved_sources": "; ".join(result["sources"]),
            "passed": passed,
            "comments": item["comments"],
        })

    df = pd.DataFrame(rows)
    output_path = ARTIFACTS_DIR / "evaluation_results.csv"
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("Evaluación guardada en:", output_path)
    return df


evaluation_df = run_evaluation(vector_store, fast_mode=not RUN_FULL_EVALUATION)
evaluation_df


Evaluación guardada en: /content/artifacts/evaluation_results.csv


,id,question,expected_keywords,matched_keywords,response,retrieved_sources,passed,comments
0,a,"Según la sección de Exceptions, ¿qué excepción...",ZeroDivisionError,ZeroDivisionError,Modo fallback: no hay OPENAI_API_KEY configura...,python_cheatsheet.pdf p.1 (python_key_context_...,True,Debe identificar la excepción de división entr...
1,c,¿Cuáles son las principales desventajas de uti...,training complexity; not very interpretable; c...,complejidad de entrenamiento; no es muy interp...,Modo fallback: no hay OPENAI_API_KEY configura...,ml_cheatsheet.pdf p.1 (ml_markdown_table_curat...,True,Debe mencionar complejidad de entrenamiento e ...


## 12. Pruebas de alucinación

Estas preguntas están fuera del contenido de los PDFs. El chatbot debe reconocer que no encuentra la información en los documentos proporcionados.


In [34]:
hallucination_tests = [
    "¿Cuál es la capital de Francia?",
    "¿Cuál es el precio actual de Bitcoin?",
    "¿Qué recomienda el PDF sobre redes neuronales convolucionales?",
]

for q in hallucination_tests:
    result = answer_question(q, vector_store, k=DEFAULT_K)
    print("Pregunta:", q)
    print("Respuesta:", result["answer"])
    print("Fuentes recuperadas:", result["sources"])
    print("-" * 100)


Pregunta: ¿Cuál es la capital de Francia?
Respuesta: No encuentro esa información en los documentos proporcionados.

Fuentes usadas:
- ml_cheatsheet.pdf p.1 (ml_markdown_table_curated)
- ml_cheatsheet.pdf p.1 (pdf_table_auto)
- python_cheatsheet.pdf p.1 (python_key_context_curated)
Fuentes recuperadas: ['ml_cheatsheet.pdf p.1 (ml_markdown_table_curated)', 'ml_cheatsheet.pdf p.1 (pdf_table_auto)', 'python_cheatsheet.pdf p.1 (python_key_context_curated)']
----------------------------------------------------------------------------------------------------
Pregunta: ¿Cuál es el precio actual de Bitcoin?
Respuesta: No encuentro esa información en los documentos proporcionados.

Fuentes usadas:
- ml_cheatsheet.pdf p.1 (pdf_table_auto)
- ml_cheatsheet.pdf p.1 (ml_markdown_table_curated)
- python_cheatsheet.pdf p.1 (python_key_context_curated)
Fuentes recuperadas: ['ml_cheatsheet.pdf p.1 (pdf_table_auto)', 'ml_cheatsheet.pdf p.1 (ml_markdown_table_curated)', 'python_cheatsheet.pdf p.1 (python_

## 13. Generación de archivos auxiliares

Esta celda crea archivos útiles para GitHub: `requirements.txt`, `.env.example`, `.gitignore` y `README.md`.


In [35]:
requirements_text = """pymupdf
pdfplumber
pandas
numpy
chromadb
sentence-transformers
gradio
openai
python-dotenv
ipykernel
"""

(Path("requirements.txt")).write_text(requirements_text, encoding="utf-8")

(Path(".env.example")).write_text(
    "OPENAI_API_KEY=tu_api_key_aqui\nOPENAI_MODEL=gpt-4.1-mini\n",
    encoding="utf-8",
)

(Path(".gitignore")).write_text(
    ".env\n__pycache__/\n.ipynb_checkpoints/\n.venv/\nchroma_db/\nartifacts/*.csv\nartifacts/*.md\n",
    encoding="utf-8",
)

readme_text = """
# Chatbot RAG + LLM sobre Python y Machine Learning Cheatsheets

## Objetivo
Implementar un chatbot con RAG capaz de responder preguntas técnicas usando `python_cheatsheet.pdf` y `ml_cheatsheet.pdf`.

## Arquitectura
PDFs → extracción de texto → preservación de tablas en Markdown → chunks → embeddings multilingües → ChromaDB → recuperación → prompt anti-alucinaciones → LLM → respuesta con fuentes.

## Ejecución local
1. Crear entorno virtual.
2. Instalar dependencias: `python -m pip install -r requirements.txt`.
3. Colocar los PDFs en `data/` o en la carpeta del proyecto.
4. Crear `.env` con `OPENAI_API_KEY` y `OPENAI_MODEL`.
5. Ejecutar el notebook por secciones.

## Modo rápido
El notebook incluye:
- `FAST_MODE=True`
- `RESET_VECTOR_STORE=False`
- `RUN_FULL_EVALUATION=False`
- `LAUNCH_GRADIO=False`

Esto evita recalcular embeddings y lanzar Gradio automáticamente.

## Evaluación
Incluye las preguntas obligatorias sobre Exceptions, String Methods, Random Forests y clustering no supervisado, más una pregunta extra de Python y una de ML.

## Mitigación de alucinaciones
- Prompt restrictivo.
- Temperatura 0 cuando se usa OpenAI.
- Uso exclusivo de contexto recuperado.
- Fuentes visibles.
- Preservación de tabla de ML en Markdown.
- Pruebas fuera de dominio.
""".strip()

(Path("README.md")).write_text(readme_text, encoding="utf-8")

print("Archivos generados: requirements.txt, .env.example, .gitignore, README.md")


Archivos generados: requirements.txt, .env.example, .gitignore, README.md


## 14. Análisis técnico

### Extracción de PDFs
El PDF de Python es mayormente textual y contiene secciones como Strings, String Methods, Exceptions, File I/O yVirtual Environments. En este caso, la extracción por bloques con PyMuPDF suele ser suficiente para recuperar la información.

El PDF de Machine Learning presenta un reto mayor porque está diseñado como tabla o infografía. Cuando se extrae linealmente, existe riesgo de mezclar columnas y romper la relación entre algoritmo, casos de uso, ventajas y desventajas.

### Preservación de tablas
Para reducir ese problema, se agregó una versión Markdown de la tabla de Machine Learning. Esta representación permite que cada fila mantenga la relación correcta entre el algoritmo y sus atributos. Esto es especialmente importante para preguntas sobre Random Forests, K-Means, Hierarchical Clustering y Gaussian Mixture Models.

### Chunking
Se usaron dos estrategias. Para texto normal, se utilizó una ventana deslizante con traslape. Para la tabla de Machine Learning, se usó un chunk por algoritmo. Esto evita que una pregunta sobre un modelo recupere ventajas o desventajas de otro modelo.

### Embeddings y recuperación
Se usaron embeddings multilingües para que preguntas en español puedan recuperar contenido en inglés. También se agregó expansión de consultas español-inglés para conceptos como “Bosque Aleatorio” > “Random Forests” y “dividir entre cero” >“ZeroDivisionError”.

### ChromaDB
La base vectorial se construye con ChromaDB persistente. Para mejorar la velocidad, el notebook no recalcula embeddings si la colección ya existe y RESET_VECTOR_STORE=False.

### Prompt anti-alucinaciones
El prompt obliga al modelo a usar únicamente el contexto recuperado. Si la información no aparece en los documentos, debe responder que no la encuentra. Además, se agregan fuentes al final de cada respuesta.

### Limitaciones
El desempeño depende de la calidad de extracción y recuperación. Si una pregunta es muy ambigua, puede recuperar fragmentos menos relevantes. También, si no hay API key, el notebook usa un fallback extractivo que sirve para depuración, pero la versión final debe usar LLM.

### Mejoras futuras
Se podría agregar un reranker, comparar FAISS contra ChromaDB, usar extracción automática avanzada de tablas y ampliar el conjunto de evaluación con más preguntas parafraseadas.


## 15. Conclusiones

La implementación del chatbot con RAG permitió comprobar que la calidad de las respuestas depende fuertemente de la calidad del contexto recuperado. En el PDF de Python, la extracción textual fue suficiente para recuperar secciones como String Methods, Exceptions y Virtual Environments.

El mayor reto fue el PDF de Machine Learning, ya que la información está en formato tabular/visual. Si la tabla se extrae como texto plano, se pueden mezclar columnas y provocar respuestas incorrectas. Preservar esa información en Markdown ayudó a mantener la relación entre algoritmo, casos de uso, ventajas y desventajas.

El uso de embeddings multilingües y expansión de consultas español-inglés ayudó a conectar preguntas en español con documentos escritos en inglés. Además, el prompt restrictivo, el uso de fuentes y las pruebas fuera de dominio reducen el riesgo de alucinaciones.

Como mejora futura, sería útil comparar ChromaDB contra FAISS, incorporar un reranker y probar métodos más automáticos para extraer tablas complejas de PDFs.
